In [ ]:
from pathlib import Path

import polars as pl
from Bio import SeqIO
from Bio.Seq import reverse_complement

In [ ]:
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory: {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
}

COVERAGE_MIN = 0.9  # fraction of the subgroup's ANs that must have a kept record
MIN_REL_LEN = 0.5   # records shorter than this fraction of the label's median length are dropped

In [ ]:
def load_genome(path: Path) -> str:
    """Genome sequence as an uppercase string; the FASTA must hold exactly one record."""
    records = list(SeqIO.parse(path, "fasta"))
    if len(records) != 1:
        raise ValueError(f"{path.name}: expected 1 record, found {len(records)}")
    return str(records[0].seq).upper()
 
 
def build_cds(pieces: pl.DataFrame, genome: str) -> str:
    """Join CDS pieces in file row order, each piece on its own strand.
 
    File row order is transcript order (checked for every multi-piece CDS, including
    trans-spliced rps12), so pieces are never sorted by coordinate.
    """
    parts = []
    size = len(genome)
    for start, end, strand in pieces.select("start", "end", "strand").iter_rows():
        if start > size:
            # circular genome written past the origin (e.g. rps12 exon 1 of a CDS that
            # wraps): the piece lies entirely in the next turn, so shift it back
            start, end = start - size, end - size
        if end <= size:
            piece = genome[start - 1 : end]  # GFF3 is 1-based, inclusive
        else:
            # circular genome: the piece crosses the origin (end is written past the length)
            piece = genome[start - 1 :] + genome[: end - size]
        parts.append(reverse_complement(piece) if strand == "-" else piece)
    return "".join(parts)
 
 
def choose_copy(
    copies: list[tuple[str, int, str]], target_len: float | None
) -> tuple[str, int, str, bool]:
    """Pick one copy among (ID, n_pieces, seq) for the same AN and label.
 
    Among copies whose length is divisible by 3, the one closest to the label's typical
    length (`target_len`, the median across genomes of each genome's longest in-frame
    copy of that label); ties go to the longer copy. This keeps the full gene over IR-boundary
    fragments (ycf1, ycf2) and over misannotated, overlong copies. If no copy is in
    frame, the longest copy is returned (it is then dropped as 'frame').
    Returns (ID, n_pieces, seq, all_copies_identical).
    """
    identical = len({seq for _, _, seq in copies}) == 1
    in_frame = [c for c in copies if len(c[2]) % 3 == 0]
    if not in_frame:
        cds_id, n_pieces, seq = max(copies, key=lambda c: len(c[2]))
    elif target_len is None:
        cds_id, n_pieces, seq = max(in_frame, key=lambda c: len(c[2]))
    else:
        cds_id, n_pieces, seq = min(in_frame, key=lambda c: (abs(len(c[2]) - target_len), -len(c[2])))
    return cds_id, n_pieces, seq, identical

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        fasta_dir = subg_dir / "fasta"
        cds_dir = subg_dir / "cds_tree" / "cds"
        out_dir = cds_dir / "fasta"
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"Processing {g}/{subg}")
 
        tsv = pl.read_csv(subg_dir / f"{subg}.tsv", separator="\t")
        all_ans = tsv["AN"].to_list()
        genome_len = (
            dict(zip(tsv["AN"], tsv["Genome_length"])) if "Genome_length" in tsv.columns else {}
        )
 
        # row index keeps the original file order (= transcript order) explicit
        cds = (
            pl.read_csv(cds_dir / "gff3_cds.tsv", separator="\t", infer_schema_length=0)
            .with_row_index("row")
            .with_columns(pl.col("start", "end").cast(pl.Int64))
        )
        cds = cds.filter(pl.col("AN").is_in(all_ans))  # only ANs still in the subgroup
 
        # --- build every CDS copy first: (AN, label) -> [(ID, n_pieces, seq), ...]
        copies_by_pair: dict[tuple[str, str], list[tuple[str, int, str]]] = {}
        problems = []
        for an in all_ans:
            an_rows = cds.filter(pl.col("AN") == an).sort("row")
            if an_rows.height == 0:
                problems.append(f"{an}: no CDS rows")
                continue
            genome_file = fasta_dir / f"{an}.fasta"
            if not genome_file.exists():
                problems.append(f"{an}: missing {genome_file.name}")
                continue
            genome = load_genome(genome_file)
            if an in genome_len and len(genome) != genome_len[an]:
                problems.append(f"{an}: FASTA length {len(genome)} != Genome_length {genome_len[an]}")
 
            for (cds_id,), pieces in an_rows.group_by("ID", maintain_order=True):
                if pieces["end"].max() > 2 * len(genome):
                    problems.append(f"{an} {cds_id}: coordinates beyond genome end")
                    continue
                size = len(genome)
                if ((pieces["start"] <= size) & (pieces["end"] > size)).any():
                    problems.append(f"{an} {cds_id}: note, a piece crosses the origin, joined around it")
                if (pieces["start"] > size).any():
                    problems.append(f"{an} {cds_id}: note, a piece is written past the origin, shifted back by the genome length")
                label = pieces["parent_gdt_label"][0]
                seq = build_cds(pieces, genome)
                copies_by_pair.setdefault((an, label), []).append((cds_id, pieces.height, seq))
 
        # --- typical length per label: median across genomes of each genome's longest in-frame
        # copy (one value per genome, so truncated IR copies or fragments can't pull it down)
        copy_lengths: dict[str, list[int]] = {}
        for (_, label), copies in copies_by_pair.items():
            in_frame = [len(s) for _, _, s in copies if len(s) % 3 == 0]
            if in_frame:
                copy_lengths.setdefault(label, []).append(max(in_frame))
        target_len = {
            label: float(pl.Series(lengths).median()) for label, lengths in copy_lengths.items() if lengths
        }
 
        # --- one record per (AN, label): the copy closest to the typical length
        chosen = []
        for (an, label), copies in copies_by_pair.items():
            cds_id, n_pieces, seq, identical = choose_copy(copies, target_len.get(label))
            chosen.append(
                {
                    "AN": an,
                    "label": label,
                    "n_copies": len(copies),
                    "copies_identical": identical,
                    "copy_lengths": ",".join(str(len(s)) for _, _, s in copies),
                    "chosen_ID": cds_id,
                    "n_pieces": n_pieces,
                    "length": len(seq),
                    "seq": seq,
                }
            )
 
        rec = pl.DataFrame(chosen)
 
        # --- per-record filters: reading frame, then length vs. the label median
        rec = rec.with_columns(
            pl.when(pl.col("length") % 3 != 0).then(pl.lit("frame")).otherwise(pl.lit("ok")).alias("status")
        )
        medians = (
            rec.filter(pl.col("status") == "ok")
            .group_by("label")
            .agg(pl.col("length").median().alias("median_len"))
        )
        rec = (
            rec.join(medians, on="label", how="left")
            .with_columns((pl.col("length") / pl.col("median_len")).round(3).alias("rel_len"))
            .with_columns(
                pl.when(pl.col("status") != "ok")
                .then(pl.col("status"))
                .when(pl.col("rel_len") < MIN_REL_LEN)
                .then(pl.lit("short"))
                .otherwise(pl.lit("kept"))
                .alias("status")
            )
        )
 
        # --- coverage, counted on kept records only
        n_ans = len(all_ans)
        coverage = (
            rec.filter(pl.col("status") == "kept")
            .group_by("label")
            .agg(pl.col("AN").n_unique().alias("kept_ANs"))
            .with_columns((pl.col("kept_ANs") / n_ans).round(3).alias("coverage"))
            .with_columns((pl.col("coverage") >= COVERAGE_MIN).alias("label_included"))
        )
 
        # --- QC table: every (AN, label) pair, 'absent' where the AN has no CDS for the label
        labels = rec.select("label").unique()
        grid = pl.DataFrame({"AN": all_ans}).join(labels, how="cross")
        qc = (
            grid.join(rec.drop("seq"), on=["AN", "label"], how="left")
            .with_columns(pl.col("status").fill_null("absent"))
            .join(coverage, on="label", how="left")
            .with_columns(
                pl.col("kept_ANs").fill_null(0),
                pl.col("coverage").fill_null(0.0),
                pl.col("label_included").fill_null(False),
            )
            .sort("label", "AN")
        )
        qc_file = cds_dir / "label_qc.tsv"
        qc.write_csv(qc_file, separator="\t")
 
        # --- one FASTA per included label, headers = AN
        included = coverage.filter("label_included").sort("label")["label"].to_list()
        stale = {p.stem for p in out_dir.glob("*.fasta")} - set(included)
        for label in included:
            kept = rec.filter((pl.col("label") == label) & (pl.col("status") == "kept")).sort("AN")
            with open(out_dir / f"{label}.fasta", "w") as fh:
                for an, seq in kept.select("AN", "seq").iter_rows():
                    fh.write(f">{an}\n{seq}\n")
 
        # --- summary
        print(f"  {n_ans} ANs | {labels.height} labels | {len(included)} labels with coverage >= {COVERAGE_MIN}")
        print(f"  {len(included)} FASTAs -> {out_dir.relative_to(subg_dir)}/ | QC -> {qc_file.relative_to(subg_dir)}")
        print(qc.group_by("status").agg(pl.len().alias("pairs")).sort("pairs", descending=True))
        print(
            f"  (AN, label) pairs with >1 copy: {rec.filter(pl.col('n_copies') > 1).height}"
            f" | copies not identical: {rec.filter((pl.col('n_copies') > 1) & ~pl.col('copies_identical')).height}"
        )
        print("  labels below the coverage cutoff:")
        print(coverage.filter(~pl.col("label_included")).sort("coverage", descending=True))
        if stale:
            print(f"  note: {len(stale)} FASTAs in {out_dir.relative_to(subg_dir)}/ are from labels no longer included: {sorted(stale)}")
        for p in problems:
            print(f"    {p}")

In [ ]:
# Records that were built but not used (frame or short), for a closer look
qc.filter(pl.col("status").is_in(["frame", "short"])).select(
    "AN", "label", "status", "n_copies", "copy_lengths", "chosen_ID", "n_pieces", "length", "median_len", "rel_len"
)

In [ ]:
# Pairs where the copies differ: which one was chosen, and how it compares to the median
qc.filter(pl.col("copies_identical") == False).select(  # noqa: E712
    "AN", "label", "status", "copy_lengths", "length", "median_len", "rel_len"
).sort("label", "AN")